# 🎙️ NVIDIA Nemotron-Hinglish-v12 Real-Time Live Streaming (SraVaani Exact UI)

Running on Tesla T4 GPU with FastAPI + WebSockets + Cloudflare Tunnel.
Matches your exact web interface: Dark mic button, blue visualizer wave, blinking cursor, and live typing!

In [ ]:
# Step 1: Install Dependencies
!pip install -q nemo_toolkit[asr] fastapi uvicorn websockets huggingface_hub torchaudio


In [ ]:
import os
import sys
import time
import json
import logging
import warnings
import subprocess
import threading
import re
import socket

# 1. NeMo dataloader warnings ko MUTE karein
try:
    from nemo.utils import logging as nemo_logging
    nemo_logging.setLevel(nemo_logging.ERROR)
except Exception:
    pass
logging.getLogger("nemo_logger").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torchaudio.transforms as T
import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Nemotron Hinglish Real-Time Live STT")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

asr_model = None
device = "cuda" if torch.cuda.is_available() else "cpu"
auto_prompt_idx = 101

# Exact UI (Aapke photo wala interface - SraVaani style)
HTML_CONTENT = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>nemotron | LIVE ASR V12 (Real-Time Hinglish)</title>
    <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=Noto+Sans+Devanagari:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        :root {
            --bg-color: #f1f5f9;
            --card-bg: #ffffff;
            --border-color: #e2e8f0;
            --text-main: #0f172a;
            --text-muted: #64748b;
            --accent-blue: #2563eb;
            --accent-red: #ef4444;
            --accent-green: #10b981;
            --font-family: 'Plus Jakarta Sans', 'Noto Sans Devanagari', sans-serif;
        }
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            background-color: var(--bg-color);
            color: var(--text-main);
            font-family: var(--font-family);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }
        .container {
            width: 100%;
            max-width: 900px;
            background: var(--card-bg);
            border-radius: 16px;
            border: 1px solid var(--border-color);
            box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.05);
            overflow: hidden;
            display: flex;
            flex-direction: column;
        }
        .top-nav {
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 16px 24px;
            border-bottom: 1px solid var(--border-color);
        }
        .logo-area { display: flex; align-items: center; gap: 8px; }
        .logo-bold { font-weight: 800; font-size: 1.15rem; letter-spacing: -0.5px; }
        .logo-divider { color: var(--border-color); }
        .logo-subtitle { font-size: 0.85rem; font-weight: 600; color: var(--text-muted); }
        .nav-pills { display: flex; gap: 6px; }
        .pill-btn {
            background: transparent;
            border: none;
            padding: 6px 14px;
            font-size: 0.8rem;
            font-weight: 600;
            color: var(--text-muted);
            border-radius: 9999px;
            cursor: pointer;
        }
        .pill-btn.active { background: #0f172a; color: #ffffff; }

        .main-card { padding: 24px; min-height: 280px; display: flex; flex-direction: column; }
        .card-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px; }
        .badge {
            background: #f1f5f9;
            color: var(--text-muted);
            font-size: 0.72rem;
            font-weight: 700;
            padding: 4px 10px;
            border-radius: 9999px;
        }
        .card-actions { display: flex; gap: 14px; }
        .action-link {
            background: none;
            border: none;
            color: var(--text-muted);
            font-size: 0.82rem;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 5px;
        }
        .transcript-display-wrapper { flex: 1; display: flex; align-items: center; padding: 10px 0; }
        .transcript-quote-box {
            font-size: 1.35rem;
            font-weight: 500;
            line-height: 1.6;
            color: var(--text-main);
            word-break: break-word;
        }
        .quote-mark { color: #94a3b8; font-family: serif; font-size: 1.6rem; }
        .cursor { color: var(--accent-blue); animation: blink 1s infinite; }
        @keyframes blink { 0%, 50% { opacity: 1; } 51%, 100% { opacity: 0; } }

        .visualizer-container { height: 36px; display: flex; align-items: center; justify-content: center; }
        canvas { width: 100%; max-width: 500px; height: 32px; }

        .bottom-bar {
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 16px 24px;
            border-top: 1px solid var(--border-color);
            background: #fbfcfd;
        }
        .mic-group { display: flex; align-items: center; gap: 14px; }
        .mic-button {
            width: 52px;
            height: 52px;
            border-radius: 50%;
            border: none;
            background: #1e293b;
            color: #ffffff;
            cursor: pointer;
            display: flex;
            justify-content: center;
            align-items: center;
            transition: all 0.2s ease;
        }
        .mic-button.recording { background: var(--accent-red); box-shadow: 0 4px 15px rgba(239, 68, 68, 0.4); }
        .mic-icon, .stop-icon { width: 22px; height: 22px; }
        .hidden { display: none !important; }

        .status-texts { display: flex; flex-direction: column; }
        .status-title { font-weight: 700; font-size: 0.92rem; }
        .status-desc { font-size: 0.78rem; color: var(--text-muted); }

        .stats-badge {
            display: flex;
            align-items: center;
            gap: 6px;
            background: #f1f5f9;
            padding: 6px 12px;
            border-radius: 9999px;
            font-size: 0.8rem;
            font-weight: 600;
        }
        .dot { width: 8px; height: 8px; border-radius: 50%; background: var(--accent-green); }
    </style>
</head>
<body>
    <div class="container">
        <header class="top-nav">
            <div class="logo-area">
                <span class="logo-bold">nemotron</span>
                <span class="logo-divider">|</span>
                <span class="logo-subtitle">LIVE ASR V12</span>
            </div>
            <div class="nav-pills">
                <button class="pill-btn active">Real-Time (80ms)</button>
                <button class="pill-btn">Instant (0ms)</button>
                <button class="pill-btn">Balanced (480ms)</button>
                <button class="pill-btn">Precise (1040ms)</button>
            </div>
        </header>

        <main class="main-card">
            <div class="card-header">
                <span class="badge">LIVE TRANSCRIPT</span>
                <div class="card-actions">
                    <button class="action-link" id="copy-btn">Copy</button>
                    <button class="action-link" id="clear-btn">Clear</button>
                </div>
            </div>

            <div class="transcript-display-wrapper">
                <div class="transcript-quote-box">
                    <span class="quote-mark">“</span>
                    <span id="transcript-text">Tap the microphone and speak in Hindi, English, or Hinglish...</span>
                    <span class="cursor" id="cursor">|</span>
                    <span class="quote-mark">”</span>
                </div>
            </div>

            <div class="visualizer-container">
                <canvas id="waveform-canvas" width="500" height="32"></canvas>
            </div>
        </main>

        <footer class="bottom-bar">
            <div class="mic-group">
                <button class="mic-button" id="mic-btn">
                    <svg class="mic-icon" viewBox="0 0 24 24" fill="currentColor">
                        <path d="M12 14c1.66 0 3-1.34 3-3V5c0-1.66-1.34-3-3-3S9 3.34 9 5v6c0 1.66 1.34 3 3 3z"/>
                        <path d="M17 11c0 2.76-2.24 5-5 5s-5-2.24-5-5H5c0 3.53 2.61 6.43 6 6.92V21h2v-3.08c3.39-.49 6-3.39 6-6.92h-2z"/>
                    </svg>
                    <svg class="stop-icon hidden" viewBox="0 0 24 24" fill="currentColor">
                        <rect x="6" y="6" width="12" height="12" rx="2"></rect>
                    </svg>
                </button>
                <div class="status-texts">
                    <span class="status-title" id="status-title">Tap the mic to speak</span>
                    <span class="status-desc" id="status-desc">Listening... Speak in Hindi, English, Hinglish...</span>
                </div>
            </div>

            <div class="stats-badge">
                <span class="dot"></span>
                <span>Live (80ms)</span>
            </div>
        </footer>
    </div>

    <script>
        let websocket = null;
        let audioContext = null;
        let mediaStream = null;
        let scriptProcessor = null;
        let analyser = null;
        let isRecording = false;

        const micBtn = document.getElementById('mic-btn');
        const micIcon = document.querySelector('.mic-icon');
        const stopIcon = document.querySelector('.stop-icon');
        const statusTitle = document.getElementById('status-title');
        const statusDesc = document.getElementById('status-desc');
        const transcriptText = document.getElementById('transcript-text');
        const canvas = document.getElementById('waveform-canvas');
        const canvasCtx = canvas.getContext('2d');

        function drawWaveform() {
            if (!isRecording) {
                canvasCtx.clearRect(0, 0, canvas.width, canvas.height);
                return;
            }
            requestAnimationFrame(drawWaveform);
            const dataArray = new Uint8Array(analyser.frequencyBinCount);
            analyser.getByteFrequencyData(dataArray);

            canvasCtx.clearRect(0, 0, canvas.width, canvas.height);
            const barWidth = 4;
            const gap = 3;
            const numBars = Math.floor(canvas.width / (barWidth + gap));
            let x = 0;

            for (let i = 0; i < numBars; i++) {
                const val = dataArray[i % dataArray.length] / 255.0;
                const barHeight = Math.max(3, val * 28);
                const y = (canvas.height - barHeight) / 2;
                canvasCtx.fillStyle = '#2563eb';
                canvasCtx.beginPath();
                canvasCtx.roundRect(x, y, barWidth, barHeight, 2);
                canvasCtx.fill();
                x += barWidth + gap;
            }
        }

        document.getElementById('copy-btn').addEventListener('click', () => {
            navigator.clipboard.writeText(transcriptText.innerText.trim());
            statusTitle.innerText = "Copied!";
            setTimeout(() => { statusTitle.innerText = isRecording ? "Listening..." : "Tap the mic to speak"; }, 1500);
        });

        document.getElementById('clear-btn').addEventListener('click', () => {
            transcriptText.innerText = "";
        });

        micBtn.addEventListener('click', async () => {
            if (isRecording) {
                stopRecording();
            } else {
                await startRecording();
            }
        });

        async function startRecording() {
            try {
                mediaStream = await navigator.mediaDevices.getUserMedia({
                    audio: { channelCount: 1, echoCancellation: true, noiseSuppression: true, autoGainControl: true }
                });

                audioContext = new (window.AudioContext || window.webkitAudioContext)();
                if (audioContext.state === 'suspended') await audioContext.resume();

                const wsProtocol = window.location.protocol === 'https:' ? 'wss:' : 'ws:';
                websocket = new WebSocket(`${wsProtocol}//${window.location.host}/ws/transcribe`);
                websocket.binaryType = "arraybuffer";

                websocket.onopen = () => {
                    websocket.send(JSON.stringify({ action: "start", sampleRate: audioContext.sampleRate }));

                    const source = audioContext.createMediaStreamSource(mediaStream);
                    analyser = audioContext.createAnalyser();
                    analyser.fftSize = 64;
                    source.connect(analyser);

                    scriptProcessor = audioContext.createScriptProcessor(2048, 1, 1);
                    source.connect(scriptProcessor);

                    const mute = audioContext.createGain();
                    mute.gain.value = 0.0;
                    scriptProcessor.connect(mute);
                    mute.connect(audioContext.destination);

                    scriptProcessor.onaudioprocess = (e) => {
                        if (!isRecording || websocket.readyState !== WebSocket.OPEN) return;
                        websocket.send(new Float32Array(e.inputBuffer.getChannelData(0)));
                    };

                    isRecording = true;
                    micBtn.classList.add('recording');
                    micIcon.classList.add('hidden');
                    stopIcon.classList.remove('hidden');
                    statusTitle.innerText = "Listening...";
                    statusDesc.innerText = "Speak now in Hindi, English, Hinglish...";
                    transcriptText.innerText = "";
                    drawWaveform();
                };

                websocket.onmessage = (event) => {
                    const data = JSON.parse(event.data);
                    if (data.type === 'partial' || data.type === 'final') {
                        if (data.text && data.text.trim().length > 0) {
                            transcriptText.innerText = data.text;
                        }
                    }
                };

                websocket.onerror = () => { stopRecording(); };
                websocket.onclose = () => { stopRecording(); };

            } catch (err) {
                alert("Microphone Error: " + err.message);
            }
        }

        function stopRecording() {
            isRecording = false;
            micBtn.classList.remove('recording');
            micIcon.classList.remove('hidden');
            stopIcon.classList.add('hidden');
            statusTitle.innerText = "Tap the mic to speak";
            statusDesc.innerText = "Ready to listen...";

            if (websocket && websocket.readyState === WebSocket.OPEN) {
                websocket.send(JSON.stringify({ action: "stop" }));
                setTimeout(() => websocket.close(), 300);
            }
            if (mediaStream) mediaStream.getTracks().forEach(t => t.stop());
            if (audioContext) audioContext.close();
        }
    </script>
</body>
</html>
"""

@app.get("/")
async def get_index():
    return HTMLResponse(content=HTML_CONTENT)

# 2. Direct GPU Tensor Inference (No disk I/O, No Lhotse dataloader bug)
@torch.inference_mode()
def run_live_inference(wav_np):
    if asr_model is None or len(wav_np) < 3200:  # < 0.2s of audio
        return ""
    try:
        sig = torch.tensor(wav_np, dtype=torch.float32, device=device).unsqueeze(0)
        sig_len = torch.tensor([len(wav_np)], dtype=torch.long, device=device)
        prompt_idx = torch.tensor([auto_prompt_idx], dtype=torch.long, device=device)

        # Fast forward pass conditioned on 'auto' language prompt
        enc, enc_len = asr_model.forward(
            input_signal=sig,
            input_signal_length=sig_len,
            prompt_indices=prompt_idx
        )

        # Fast RNN-T greedy decoding
        res = asr_model.decoding.rnnt_decoder_predictions_tensor(
            encoder_output=enc,
            encoded_lengths=enc_len,
            return_hypotheses=False
        )

        if isinstance(res, tuple):
            res = res[0]
        if isinstance(res, list) and len(res) > 0:
            first = res[0]
            if hasattr(first, 'text'):
                return first.text.strip()
            return str(first).strip()
        elif hasattr(res, 'text'):
            return res.text.strip()
        return str(res).strip()
    except Exception as e:
        print(f"[!] Direct inference error: {e}")
    return ""

@app.websocket("/ws/transcribe")
async def websocket_transcribe(websocket: WebSocket):
    await websocket.accept()
    client_sr = 16000
    resampler = None
    audio_buffer = np.array([], dtype=np.float32)
    last_process_time = time.time()

    try:
        while True:
            message = await websocket.receive()
            if "text" in message:
                data = json.loads(message["text"])
                action = data.get("action")
                if action == "start":
                    client_sr = int(data.get("sampleRate", 16000))
                    audio_buffer = np.array([], dtype=np.float32)
                    last_process_time = time.time()
                    resampler = T.Resample(client_sr, 16000) if client_sr != 16000 else None
                    await websocket.send_text(json.dumps({"type": "ready"}))
                    print(f"[*] Live Stream Connected (Client SR: {client_sr} Hz)")

                elif action == "stop":
                    if len(audio_buffer) >= (client_sr * 0.3) and asr_model is not None:
                        if resampler is not None:
                            wav_t = resampler(torch.from_numpy(audio_buffer).unsqueeze(0)).squeeze(0)
                            wav_data = wav_t.numpy()
                        else:
                            wav_data = audio_buffer
                        final_text = run_live_inference(wav_data)
                        if final_text:
                            print(f"[FINAL] Transcribed: \"{final_text}\"")
                            await websocket.send_text(json.dumps({"type": "final", "text": final_text}))
                    break

            elif "bytes" in message and asr_model is not None:
                pcm_block = np.frombuffer(message["bytes"], dtype=np.float32)
                audio_buffer = np.concatenate([audio_buffer, pcm_block])

                now = time.time()
                # Run ASR inference every ~300ms once we have at least 0.5s of audio
                if (now - last_process_time) >= 0.30 and len(audio_buffer) >= (client_sr * 0.5):
                    last_process_time = now
                    if resampler is not None:
                        wav_t = resampler(torch.from_numpy(audio_buffer).unsqueeze(0)).squeeze(0)
                        wav_data = wav_t.numpy()
                    else:
                        wav_data = audio_buffer

                    partial_text = run_live_inference(wav_data)
                    if partial_text:
                        print(f"[LIVE] \"{partial_text}\"")
                        await websocket.send_text(json.dumps({"type": "partial", "text": partial_text}))

    except Exception as e:
        pass

# 3. Load Model
import nemo.collections.asr as nemo_asr
from huggingface_hub import hf_hub_download

print("🚀 1. Loading Nemotron-Hinglish-v12 on Tesla T4 GPU...")
model_path = hf_hub_download(repo_id="smajji/nemotron-hinglish-v12", filename="nemotron-hinglish-v12.nemo")
asr_model = nemo_asr.models.ASRModel.restore_from(model_path, map_location=device)
asr_model.eval()

# Prompt language index setup (Auto: 101)
prompt_dict = getattr(asr_model.cfg.model_defaults, 'prompt_dictionary', {})
if hasattr(prompt_dict, 'get'):
    auto_prompt_idx = prompt_dict.get('auto', 101)
else:
    auto_prompt_idx = 101

print(f"✅ Model Loaded on {device.upper()}! Language prompt index set to: {auto_prompt_idx} ('auto')")

# 4. Find free port dynamically & Run FastAPI in background thread
def find_free_port(start=8000):
    for p in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', p)) != 0:
                return p
    return 8080

PORT = find_free_port(8000)
print(f"[*] Selected Free Port: {PORT}")

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

print(f"⏳ Waiting for FastAPI server to initialize on port {PORT}...")
time.sleep(3)
print(f"✅ Server is UP and listening on port {PORT}!")

# 5. Start Cloudflare Tunnel
print("🌐 Starting Cloudflare Tunnel...")
os.system("pkill -f cloudflared")
os.system("curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(40):
    line = proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.3)

print("\n" + "="*65)
print(f" 🌟 OPEN THIS IN YOUR BROWSER (EXACT PHOTO UI):")
print(f" 👉 {tunnel_url}")
print("="*65 + "\n")

# Keep Colab cell alive
while True:
    time.sleep(1)


## 📁 Test Any Audio File (e.g. Kotak Bank Call)

Run this cell to transcribe any `.wav` or `.mp3` file.
If `kotak_call.wav` is not found, a **Choose Files** upload button will automatically appear!

In [ ]:
# =====================================================================
# 🎙️ NVIDIA Nemotron-Hinglish-v12: High-Accuracy Call Transcriber
# =====================================================================
import os
import math
import torch
import torchaudio
import torchaudio.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Audio file check
audio_path = "20260710-172448_7575030388_2013-all (1).wav"
if not os.path.exists(audio_path):
    audio_path = "20260710-172448_7575030388_2013-all.wav"
if not os.path.exists(audio_path):
    audio_path = "kotak_call.wav"

print(f"🎵 Using Audio File: {audio_path}")

# 2. Load and Resample to 16kHz Mono
sig, sr = torchaudio.load(audio_path)
if sig.shape[0] > 1:
    sig = sig.mean(dim=0, keepdim=True)
if sr != 16000:
    sig = T.Resample(sr, 16000)(sig)
    sr = 16000

sig = sig.to(device)
total_samples = sig.shape[1]
duration_sec = total_samples / 16000
print(f"⏱️ Total Duration: {duration_sec:.2f}s ({duration_sec/60:.2f} minutes)\n")

# 3. FastConformer 5-Second Conversational Slices
# FastConformer streaming has an optimal chunk size of 5-8 seconds per turn
CHUNK_SEC = 6.0
chunk_samples = int(CHUNK_SEC * 16000)
num_chunks = math.ceil(total_samples / chunk_samples)
prompt_idx = torch.tensor([101], dtype=torch.long, device=device)  # 101 = auto (90% training)

transcriptions = []
print(f"⚡ Transcribing {num_chunks} dialogue turns (6s each) with Nemotron-v12...\n")

with torch.inference_mode():
    for i in range(num_chunks):
        start = i * chunk_samples
        end = min((i + 1) * chunk_samples, total_samples)
        c_sig = sig[:, start:end]
        
        # Skip near-silent chunks
        if torch.max(torch.abs(c_sig)) < 0.02:
            continue
            
        c_len = torch.tensor([c_sig.shape[1]], dtype=torch.long, device=device)
        
        enc, enc_len = asr_model.forward(
            input_signal=c_sig,
            input_signal_length=c_len,
            prompt_indices=prompt_idx
        )
        res = asr_model.decoding.rnnt_decoder_predictions_tensor(
            encoder_output=enc,
            encoded_lengths=enc_len,
            return_hypotheses=False
        )
        
        if isinstance(res, tuple):
            res = res[0]
        if isinstance(res, list) and len(res) > 0:
            first = res[0]
            txt = first.text.strip() if hasattr(first, 'text') else str(first).strip()
        elif hasattr(res, 'text'):
            txt = res.text.strip()
        else:
            txt = str(res).strip()
            
        t_start = i * CHUNK_SEC
        t_end = min((i + 1) * CHUNK_SEC, duration_sec)
        if txt:
            m_s = int(t_start // 60)
            s_s = int(t_start % 60)
            m_e = int(t_end // 60)
            s_e = int(t_end % 60)
            time_tag = f"[{m_s:02d}:{s_s:02d} - {m_e:02d}:{s_e:02d}]"
            print(f"{time_tag} {txt}")
            transcriptions.append(f"{time_tag} {txt}")

full_transcript = "\n".join(transcriptions)

print("\n" + "="*70)
print(f"🎯 COMPLETE DIALOGUE TRANSCRIPTION ({len(transcriptions)} TURNS):")
print("="*70)
print(full_transcript)
print("="*70)

output_file = "nemotron_complete_call_transcript.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(full_transcript)

print(f"\n💾 Saved {len(full_transcript)} characters to '{output_file}'!")


## 🇮🇳 Indic-Transcribe-Core (1.2B by Bodhan AI & IIT Madras)

Trained specifically on Indian telephony, regional accents, and Hinglish.
Includes built-in `long_form.py` which splits long call center audio at natural pauses (<=30s).

In [ ]:
# =====================================================================
# 🇮🇳 Indic-Transcribe-Core (1.2B) by Bodhan AI & IIT Madras
# =====================================================================
!pip install -q transformers sentencepiece soundfile huggingface_hub torch torchaudio

import os
import sys
from huggingface_hub import snapshot_download

# Verified Hugging Face Token
HF_TOKEN = "your_huggingface_token_here"

# 1. Download Model Snapshot (Model code is included in repo)
print("🚀 Downloading bodhan-ai/indic-transcribe-core (1.2B params)...")
model_dir = snapshot_download("bodhan-ai/indic-transcribe-core", token=HF_TOKEN)
sys.path.insert(0, model_dir)

# 2. Load Model on GPU
from indic_transcribe import IndicTranscribe
from long_form import transcribe_long

print("⚡ Loading IndicTranscribe model on GPU...")
asr = IndicTranscribe.from_pretrained(model_dir, device="cuda")

# 3. Audio File Check
audio_path = "20260710-172448_7575030388_2013-all (1).wav"
if not os.path.exists(audio_path):
    audio_path = "20260710-172448_7575030388_2013-all.wav"
if not os.path.exists(audio_path):
    audio_path = "kotak_call.wav"

print(f"🎙️ Transcribing {audio_path} with Long-Form Silence Detection (Hindi Mode)...")

# 4. Run Long-form transcription (IIT Madras pause-based chunking)
full_transcript = transcribe_long(asr, audio_path, lang="hi")

print("\n" + "=" * 70)
print("🎯 INDIC-TRANSCRIBE-CORE (BODHAN AI / IIT MADRAS) OUTPUT:")
print("=" * 70)
print(full_transcript)
print("=" * 70)

# 5. Save transcript file
with open("indic_transcribe_core_transcript.txt", "w", encoding="utf-8") as f:
    f.write(full_transcript)

print("\n💾 Complete transcript saved to 'indic_transcribe_core_transcript.txt'!")
